# QF 627 Programming and Computational Finance
## Lesson 08 | Mini-Mock Assessment: Individual Practice Session

***

> Good afternoon, Team 👋

> The purpose of the current assessment is to serve as a simulation of the actual assessment day. Its main goal is simple—to help you prepare effectively and to shed light on your current level of knowledge and expertise.

> Through this `mock` `mini-assessment`, you’ll get a chance to see how questions might be structured and how you may want to approach writing your answers on the actual assessment day. Please note that this is not the real assessment, and it will not be graded.

> Importantly, this exercise will help me better understand where each of you stands, so I can support you more effectively during our final two weeks of learning.

> Don’t feel pressured—the actual assessment will give you three hours, while this mini mock assessment will take only 70 minutes.

***

> Be sure to submit your work before the deadline: `3:10pm, November 21, 2025`. It is an open-book exercise, and is also a timed task. To be fair to all students, a late submission will incur a point reduction.

> Please note `your last name` for `naming your submission` file (e.g., `Roh.ipynb`)

> If you find that you cannot answer a question, it would be wise to move on to another question that you can answer, and to finish that one first. `Make the best use of the time available`. If you cannot fully answer all the questions, then do as much as you can.

***

> Rather than feeling pressured by the assessment, I hope you will enjoy the opportunity presented by the hands-on exercise. You will notice that `answering each question will further consolidate your learning`.

***

> I wish you the best for your individual assessment 🤞

***

### For standardization of your answers…

> Please execute the lines of code below before you start work on your answers.

In [1]:
# Our standardized printing options

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

np.set_printoptions(precision = 3)

pd.set_option("display.float_format", lambda x: "%.3f" % x)

plt.style.use("ggplot")

mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.alpha"] = 0.25

mpl.rcParams["axes.facecolor"] = "white"

mpl.rcParams["legend.fontsize"] = 14

List of Questions

    IMPORTANT NOTE: 

### <font color = purple> <center> One of the key aspects of the current assessment involves data wrangling. 

### <font color = purple> <center> Please ensure that you correctly wrangle your data to obtain valid answers.

### <font color = purple> <center> When using a function, please write the function within the current script.

##  <font color = blue> 👉 Question 1. </font> Please use the datasets `qs_A.csv`, `qs_B.csv`, `qs_C.csv`. The time period for analysis is from October 2006 to December 2012.

### The strategy that you'll be developing is as follows: you create two separate Simple Moving Averages (SMA) of a time series with differing lookback periods (here, 40 days and 100 days). If the short moving average exceeds the long moving average then you go long, if the long moving average exceeds the short moving average then you exit.

```python
    df["Signal"] = 0
    df.loc[df["SMA_40"] > df["SMA_100"], "Signal"] = 1
    df.loc[df["SMA_40"] < df["SMA_100"], "Signal"] = 0
```

### On the days that the signal is 1 and the short moving average crosses the long moving average (for the period greater than the shortest moving average window), you'll buy a 100 shares. The days on which the signal is 0, the final result will be 0 as a result of the operation 100 x signal.

### For rolling statistics, set `min_periods` at `1` and `center` argument at `False`.

### Use `Adj Close` price.

### Let’s suppose that you started from a `$100,000` capital base for each of the three securities.

### Disregarding commission, how much will you have in the end in your account for each of the securities as a result of the current momentum-based trading?

### Below are the lines of code that lead to an answer:

### <font color = red> Answer 1 </font>


    A  : _$______________ 
    
    B  : _$______________ 
    
    C  : _$______________ 


In [2]:
def download_csv_data(file_path):
    """
    Download stock data from a CSV file
    """
    stock_data = pd.read_csv(file_path, 
                             index_col=0, 
                             parse_dates=True)
    return stock_data
# stock_data = download_csv_data('alphas.csv')

In [42]:
def get_momentum_strategy(df, sma_ls, shares, position_shift=1):
    """
    Apply single stock momentum strategy pipeline to price DataFrame `df`. 
    Price is in first column.
    Adds columns:
      positions, trade, passive_returns, strategy_returns,
      cum_returns, cum_strategy_returns
    Parameters:
      df        : pd.DataFrame with price columns for ticker name as the header
      sma_tuple : tuple of (short_window, long_window) for SMAs
    Returns:
      pd.DataFrame with the above columns
    """
    def get_pnl(df, shares):
        df['pnl'] = (df[df.columns[0]].diff() * shares * df['positions']).fillna(0)
        # print(df)
        return df
    def get_sma(df, sma_list):
        """
        Take DF and given list of SMA to enrich df when stock price is in first column
        """
        for window in sma_list:
            df[f"sma_{window}"] =\
            (
                df[df.columns[0]]
                .rolling(window = window,
                         min_periods = 1,
                         center = False)
                .mean()
            )

        return df
    if sma_ls[0] >= sma_ls[1]:
        raise ValueError("sma_short must be less than sma_long")
    df = get_sma(df, sma_ls).dropna()

    df['positions'] =\
    (
        np.where((df[f'sma_{sma_ls[0]}'] > df[f'sma_{sma_ls[1]}']), 1, 0)
    )
    df['positions'] = df['positions'].shift(position_shift).fillna(0)
    df['trade'] = \
    (
        df['positions'].diff().fillna(0)
    )
    if df.at[df.index[0], 'positions'] != 0:
        df.at[df.index[0], 'trade'] = df.at[df.index[0], 'positions']
    df['passive_returns'] =\
    (    # get passive returns
        np.log(df[df.columns[0]] 
            / df[df.columns[0]].shift(1))
    ).fillna(0)
    df['strategy_returns'] =\
    (     # get strategy returns
        df['passive_returns'] * df['positions'].shift(1).fillna(0)
    )
    df['cum_returns'] =\
    (
        df['passive_returns'].cumsum().apply(np.exp).fillna(1)
    )
    df['cum_strategy_returns'] =\
    (
        df['strategy_returns'].cumsum().apply(np.exp).fillna(1)
    )
    df = get_pnl(df, shares)

    return df

# sma_list = [42, 252]
# get_momentum_strategy(df[['GS']], sma_list)

In [47]:
tickers = ['A', 'B', 'C']
sma_ls = [40, 100]
shares_ = 100
init_cap = 100_000

stock_dct = {}

for ticker in tickers:
    df = download_csv_data(f'qs_{ticker}.csv')[['Adj Close']]
    df = get_momentum_strategy(df, sma_ls, shares_)
    print(f"{ticker}: ${df.pnl.sum() + init_cap:,.2f}")

    stock_dct[ticker] = df

A: $100,900.54
B: $100,316.34
C: $99,315.41


##  <font color = blue> 👉 Questions 2 & 3. </font>  

### Calculate and visualize the maximum drawdowns and the longest drawdown periods for `A`, `B`, and `C`.

### Below are the lines of code that lead to an answer:

In [57]:
from lets_plot import *
LetsPlot.setup_html()

def get_drawdowns_plot(df):
    df['daily_drawdown'] = df['cum_returns'] / df['cum_returns'].cummax() - 1
    # take date column out and cumsum for periods
    dd_reset = df.reset_index()
    dd_reset['period'] = (dd_reset['daily_drawdown'] == 0).cumsum()
    # find nodes with 0 value i.e. highpoints
    dd_nonzero = dd_reset[dd_reset['daily_drawdown'] != 0]
    # aggregate by period
    period_stats = dd_nonzero.groupby('period').agg(
        start_date = ('Date', 'min'),
        end_date = ('Date', 'max'),
        avg_dd = ('daily_drawdown', 'mean'),
        max_dd = ('daily_drawdown', 'min'),
        duration=('daily_drawdown', 'count')
    ).sort_values(by='avg_dd',ascending=True)


    ## printing drawdowns   
    print(f'      As to {ticker},')
    print(f"    The maximum drawdown is about {-period_stats.max_dd.min():.2%} percentage points.")
    print(f"    The longest drawdown period lasts for {period_stats.duration.max():.0f} days.")


    # plot drawdowns
    # from lets_plot import *
    # LetsPlot.setup_html()
    p =\
    (
        ggplot(dd_reset, aes(x='Date')) +
        geom_line(aes(y='daily_drawdown'), color='grey', size=0.7) +
        # geom_line(aes(y='sma_50'), color='orange', size=0.7,linetype=2) +   # amend sma as required
        # geom_line(aes(y='sma_200'), color='blue', size=0.7,linetype=2) +
        # geom_point(aes(y='sma_50'), data=wmt_melt[wmt_melt['trade']>0.0], color='red', size=3) +       # amend trade as required
        # geom_point(aes(y='sma_50'), data=wmt_melt[wmt_melt['trade']<-0.0], color='blue', size=3) +
        ggtitle(f"Drawdown for {ticker}") +
        ylab("pct loss") +
        scale_y_continuous(format='.0%') + 
        xlab("Date") +
        ggsize(1200, 500) 
    )
    display(p)

    return period_stats

### <font color = red> Answer 2 (`visualization component`; use `lets-plot`) is presented in the cell below: </font>

### <font color = red> Answer 3 </font>
    
    As to A,
    
    The maximum drawdown is about ____________ percentage points.
    The longest drawdown period lasts for _____________ days.
    
    As to B,
    
    The maximum drawdown is about ____________ percentage points.
    The longest drawdown period lasts for _____________ days.
    
    As to C,
    
    The maximum drawdown is about ____________ percentage points.
    The longest drawdown period lasts for _____________ days.


In [ ]:
for ticker, df in stock_dct.items():
    get_drawdowns_plot(df)
    

      As to A,
    The maximum drawdown is about 60.87% percentage points.
    The longest drawdown period lasts for 456 days.


      As to B,
    The maximum drawdown is about 65.29% percentage points.
    The longest drawdown period lasts for 1229 days.


      As to C,
    The maximum drawdown is about 57.94% percentage points.
    The longest drawdown period lasts for 1297 days.


###  <font color = blue> 👉 Question 4. </font> Using the current momentum strategy, which of the securities shows the greatest Sharpe ratio?

### Below are the lines of code that lead to an answer:

In [60]:
def get_sharpe(daily_returns):
    """
    get_sharpe Calculate the annualized Sharpe ratio from a series of daily returns.
    Annualized Sharpe ratio computed as:
                sqrt(252) * mean(daily_returns) / std(daily_returns)
    - Assumes 252 trading days per year for annualization. Adjust the
      multiplier for a different convention.
    - Input should be returns (not prices). Convert prices to returns before
      calling this function.
    Parameters:
        daily_returns : array-like (pd.Series or np.ndarray)
    Returns:
        float
    Usage:
        get_sharpe_ratio(df['strategy_returns'])
    """
    return np.sqrt(252) * daily_returns.mean() / daily_returns.std(ddof = 1)

# get_sharpe_ratio(ibm['strategy_returns'])
# print(f"The answer is {max(sharpe_dct, key=s_dct.get).upper()} with  {max(sharpe_dct.values()):.5f}")

In [65]:
sharpe = {
    ticker: get_sharpe(df['strategy_returns']) for ticker, df in stock_dct.items() 
}

In [66]:
sharpe

{'A': np.float64(0.8689126085823347),
 'B': np.float64(0.21330027103389163),
 'C': np.float64(-0.37713145010955507)}

### <font color = red> Answer 4 </font>

    The answer is ____________________________ .

In [67]:
print(f"The answer is {max(sharpe, key=sharpe.get)}")

The answer is A


###  <font color = blue> 👉 Question 5. </font> Report compound annual growth rate (CAGR) for `A`, `B`, and `C`.

### Below are the lines of code that lead to an answer:

In [68]:
def get_cagr(cumulative_returns, annual_days=365.25):
    """
    get_cagr Compute Compound Annual Growth Rate (CAGR) from a series of cumulative returns.
        NOTE: please ensure index is date
    Parameters:
        cumulative_returns : pandas.Series. Time-indexed series of cumulative returns (e.g. cumulative growth factors)
    Returns:
        float
    """

    cumulative_returns = cumulative_returns.dropna()
    n_of_days = (cumulative_returns.index[-1] - cumulative_returns.index[0]
                ).days
    cagr =\
    (
        (        
        cumulative_returns.iloc[-1] 
        /
        cumulative_returns.iloc[0]
        ) ** (annual_days / n_of_days)
        - 1
    )
    return cagr

# print(f"The answer is {max(c_dct, key=c_dct.get).upper()} with {max(c_dct.values()):.2%}")

In [69]:
cagr = {
    ticker: get_cagr(df['cum_strategy_returns']) for ticker, df in stock_dct.items() 
}

### <font color = red> Answer 5 </font>


    A  : _____________%__ 
    
    B : _____________%__ 
    
    C  : _____________%__ 


In [71]:
for ticker, cagr_ in cagr.items():
    print(f"{ticker}: {cagr_:.2%}")

A: 27.44%
B: 4.53%
C: -6.18%


    IMPORTANT NOTE: 

### <font color = purple> <center> Prior to submitting, ensure that you execute the following command to present your workspace.

### <font color = purple> <center> Before submission, ensure that your responses are entered into the designated cells provided for answering.

In [72]:
%whos

Variable                     Type         Data/Info
---------------------------------------------------
GGBunch                      type         <class 'lets_plot.plot.plot.GGBunch'>
LetsPlot                     type         <class 'lets_plot.LetsPlot'>
aes                          function     <function aes at 0x1252d4680>
arrow                        function     <function arrow at 0x125b4f920>
as_discrete                  function     <function as_discrete at 0x125b4ca40>
cagr                         dict         n=3
cagr_                        float64      Shape: ()
coord_cartesian              function     <function coord_cartesian at 0x125aa65c0>
coord_fixed                  function     <function coord_fixed at 0x125aa6660>
coord_flip                   function     <function coord_flip at 0x125aa67a0>
coord_map                    function     <function coord_map at 0x125aa6700>
coord_polar                  function     <function coord_polar at 0x125aa6840>
df                  

### <font color = green> 💯 Thank you for putting your efforts into our mini-mock individual assessment questions 😊